In [49]:
tabla='qtsoo10'
col_tabla = 'solopefec'
essi='essi_dat_cqx007_etl'
col_essi='sol_fec'
dat= "dat_ceqx007_essi"
col_dat='sol_fec'

In [50]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import numpy as np

In [51]:
inicioTiempo = time.time()
inicioProceso=time.time()
now_inicio = datetime.now()

In [52]:
######################FUNCIONES DE LOG###########################
global dim, control_a, control_d, base1, falla, merge
control_a=[]
control_d=[]
dim=[]
falla=[]
id_afectado=[]

def log_falla(id, cod, isNeeded):
    if isNeeded:
        filas_perdidas = merge.loc[pd.isnull(merge[id])]
        filas_perdidas = filas_perdidas.drop_duplicates(subset=[cod])
        filas_perdidas=filas_perdidas[[cod]]
        if filas_perdidas.empty:
            filas_perdidas_string = 0
        else:
            filas_perdidas_string = filas_perdidas.to_string(index=False, header=False)
            filas_perdidas_string = filas_perdidas_string.replace('\n', ',')
    else:
        filas_perdidas_string = 0
    falla.append(filas_perdidas_string)
    id_afectado.append(id)

def log_control(table):    
    dim.append(table)
    control_d.append(base1.shape[0])
    control_a.append(base1.shape[0])

In [53]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()


In [54]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)
now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [55]:

fecha= '2023-05-01'

In [56]:
cas = pd.read_sql_query(f"SELECT id_cas,cod_cas FROM dim_cas where id_cas is not null", con=connection2)
cas = cas.drop_duplicates(subset=['cod_cas'])
cas=cas.dropna()
red = pd.read_sql_query(f"SELECT id_red,cod_red FROM dim_red", con=connection2)

oricas = pd.read_sql_query(f"SELECT id_oricas,ori_cod FROM dim_oricas", con=connection2)
oricas["ori_cod"]=oricas["ori_cod"].str.strip()

cps= pd.read_sql_query(f"SELECT id_cps,cod_cps FROM dim_cps", con=connection2)
cps["cod_cps"]=cps["cod_cps"].str.strip()

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)

pacsec = pd.read_sql_query(f"SELECT id_pacsec,per_sec FROM dim_pacsec", con=connection2)


In [57]:
base1=pd.read_sql_query(f"SELECT * FROM {essi} WHERE {col_essi} >='{fecha}'", con=connection1)

#INICIO DEL DAT

In [58]:
base1.shape

(100216, 15)

In [59]:
#Eliminamos las columnas que no se usarán aquí: las descripciones previa evaluación

# Lista de columnas a eliminar
columnas_eliminar = ['des_cas', 'des_red', 'grd_com']

# Eliminar las columnas
base1 = base1.drop(columns=columnas_eliminar)

In [60]:
base1.shape

(100216, 12)

In [61]:
inicioProceso=time.time()

In [62]:
base2=pd.read_sql_query(f"SELECT * FROM {dat} LIMIT 10", con=connection2)

In [63]:
base1.shape

(100216, 12)

In [64]:
control_a.append(base1.shape[0])

In [65]:
oricas=oricas.rename(columns={"ori_cod":"ori_cas"})
base1['ori_cas']=base1['ori_cas'].str.strip()
base1["ori_cas"]=base1["ori_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,oricas,how='left',on='ori_cas')
base1=pd.merge(base1,oricas,how='inner',on='ori_cas')
base1.shape

(100216, 13)

In [66]:
log_falla('id_oricas', 'ori_cas', True)
log_control('dim_oricas')
base1=base1.drop('ori_cas',axis=1)

In [67]:
base1['cod_cas']=base1['cod_cas'].str.strip()
base1["cod_cas"]=base1["cod_cas"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cas,how='left',on='cod_cas')
base1=pd.merge(base1,cas,how='inner',on='cod_cas')
base1.shape

(100216, 13)

In [68]:
log_falla('id_cas', 'cod_cas', True)
log_control('dim_cas')
base1=base1.drop('cod_cas',axis=1)

In [69]:
base1['cod_red']=base1['cod_red'].str.strip()
base1["cod_red"]=base1["cod_red"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,red,how='left',on='cod_red')
base1=pd.merge(base1,red,how='inner',on='cod_red')
base1.shape

(100216, 13)

In [70]:
log_falla('id_red', 'cod_red', True)
log_control('dim_red')
base1=base1.drop('cod_red',axis=1)

In [71]:
merge.shape

(100216, 13)

In [72]:
base1['cod_cps']=base1['cod_cps'].str.strip()
base1["cod_cps"]=base1["cod_cps"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cps,how='inner',on='cod_cps')
base1=pd.merge(base1,cps,how='left',on='cod_cps')
base1.shape

(100216, 13)

In [73]:
merge.shape #Merge pierde todo

(100216, 13)

In [74]:
log_falla('id_cps', 'cod_cps', True)
log_control('dim_cps')
base1=base1.drop('cod_cps',axis=1)

In [75]:
cps=cps.rename(columns={"id_cps": "id_paqqui", "cod_cps":"paq_qui"})
base1['paq_qui']=base1['paq_qui'].str.strip()
base1["paq_qui"]=base1["paq_qui"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,cps,how='inner',on='paq_qui')
base1=pd.merge(base1,cps,how='left',on='paq_qui')
base1.shape

(100216, 13)

In [76]:
merge.shape #Merge pierde todo

(65458, 13)

In [77]:
log_falla('id_paqqui', 'paq_qui', True)
log_control('dim_cps')
base1=base1.drop('paq_qui',axis=1)

In [78]:
pacsec=pacsec.rename(columns={"per_sec": "pac_sec"})
pacsec['pac_sec']=pacsec['pac_sec'].astype(str).str.strip()
pacsec["pac_sec"]=pacsec["pac_sec"].str.replace(' ', '', regex=True)
base1['pac_sec']=base1['pac_sec'].astype(str).str.strip()
base1["pac_sec"]=base1["pac_sec"].str.replace(' ', '', regex=True)
merge=pd.merge(base1,pacsec,how='inner',on='pac_sec')
base1=pd.merge(base1,pacsec,how='left',on='pac_sec')
base1.shape

(100216, 13)

In [79]:
merge.shape

(98891, 13)

In [80]:
log_falla('id_pacsec', 'pac_sec', True)
base1=base1.drop('pac_sec',axis=1)
dim.append('dim_pacsec')
control_d.append(base1.shape[0])

In [81]:
base1['sol_fec_temp'] = base1['sol_fec'].astype(str).str.split().str[0]
tiempo=tiempo.rename(columns={"dt_fecha":"sol_fec_temp"})
tiempo["sol_fec_temp"] = tiempo['sol_fec_temp'].astype(str)
merge = pd.merge(base1, tiempo, how='inner', on='sol_fec_temp')
base1 = pd.merge(base1, tiempo, how='left', on='sol_fec_temp')
base1=base1.rename(columns={"id_tiempo":"id_time_sol","dt_ano":"ano_sol","dt_mes":"mes_sol",
                            "dt_dia":"dia_sol","dt_dia_sem":"dia_sem_sol","dt_sem":"sem_sol"})
base1=base1.drop("sol_fec_temp",axis=1)
base1.shape

(100216, 18)

In [82]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 100216 entries, 0 to 100215
Data columns (total 18 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   sol_num      100216 non-null  float64
 1   flg_pri      65458 non-null   float64
 2   paq_sec      65458 non-null   float64
 3   sol_med      65458 non-null   float64
 4   sol_fec      100216 non-null  object 
 5   act_med      100216 non-null  float64
 6   id_oricas    100216 non-null  int64  
 7   id_cas       100216 non-null  int64  
 8   id_red       100216 non-null  int64  
 9   id_cps       100216 non-null  int64  
 10  id_paqqui    65458 non-null   float64
 11  id_pacsec    98891 non-null   float64
 12  id_time_sol  100215 non-null  float64
 13  mes_sol      100215 non-null  float64
 14  dia_sol      100215 non-null  float64
 15  dia_sem_sol  100215 non-null  float64
 16  sem_sol      100215 non-null  float64
 17  ano_sol      100215 non-null  float64
dtypes: float64(13), int64(4)

In [83]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [84]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df1_columns - df2_columns
different_columns

set()

In [85]:
borrando=f"DELETE FROM {dat} WHERE {col_dat} >='{fecha}'"
borrado = connection2.execute(borrando)

In [86]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [87]:
base

,id_red,sol_fec,act_med,paq_sec,id_paqqui,id_cps,sol_num,sol_med,id_pacsec,id_cas,dia_sem_sol,id_time_sol,dia_sol,flg_pri,id_oricas,sem_sol,ano_sol,mes_sol
0,3,2023-05-01,13384739.0,0.0,76850.0,76850,107872.0,0.0,49508046.0,565,1.0,20230501.0,1.0,1.0,1,1.0,2023.0,5.0
1,3,2023-05-01,13250473.0,0.0,80602.0,80602,107858.0,0.0,52500264.0,565,1.0,20230501.0,1.0,1.0,1,1.0,2023.0,5.0
2,3,2023-05-01,13379565.0,0.0,82526.0,82526,107860.0,0.0,45948844.0,565,1.0,20230501.0,1.0,1.0,1,1.0,2023.0,5.0
3,3,2023-05-01,13384132.0,NaN,NaN,76850,107866.0,NaN,52312983.0,565,1.0,20230501.0,1.0,NaN,1,1.0,2023.0,5.0
4,3,2023-05-01,13383731.0,0.0,77355.0,77355,107869.0,0.0,38997573.0,565,1.0,20230501.0,1.0,1.0,1,1.0,2023.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100211,29,2023-07-03,2693121.0,0.0,79550.0,79550,22932.0,0.0,45112714.0,483,1.0,20230703.0,3.0,0.0,1,2.0,2023.0,7.0
100212,29,2023-07-03,2693298.0,0.0,85108.0,85108,22939.0,0.0,48602466.0,483,1.0,20230703.0,3.0,1.0,1,2.0,2023.0,7.0
100213,29,2023-07-04,2688950.0,0.0,79355.0,79355,22962.0,0.0,NaN,483,2.0,20230704.0,4.0,1.0,1,2.0,2023.0,7.0
100214,29,2023-07-04,2685860.0,0.0,80913.0,80913,22961.0,0.0,58210593.0,483,2.0,20230704.0,4.0,1.0,1,2.0,2023.0,7.0


In [88]:
base.columns

Index(['id_red', 'sol_fec', 'act_med', 'paq_sec', 'id_paqqui', 'id_cps',
       'sol_num', 'sol_med', 'id_pacsec', 'id_cas', 'dia_sem_sol',
       'id_time_sol', 'dia_sol', 'flg_pri', 'id_oricas', 'sem_sol', 'ano_sol',
       'mes_sol'],
      dtype='object')

In [89]:
base.to_sql(name=f'{dat}', con=connection2, if_exists='append', index=False,chunksize=10000)

10216

In [90]:
fecha_fin = "2024-12-31"

In [91]:
proceso = "DAT"
cod_proceso = 13

# proceso = pd.read_sql_query("SELECT des_mod FROM etl_act where id_mod=13", con=connection2)
# proceso = proceso.iloc[0, 0]
# cod_proceso = pd.read_sql_query("SELECT id_mod FROM etl_act where id_mod=13", con=connection2)
# cod_proceso = cod_proceso.iloc[0, 0]

now_fin = datetime.now()
fecha_log = now.strftime("%Y-%m-%d")
hora_log_inicio = now_inicio.strftime("%H:%M")
hora_log_fin = now_fin.strftime("%H:%M")
tabla_logs = pd.DataFrame({'esperado':control_a,'obtenido':control_d,'dim':dim,'falla':falla})
tabla_logs['proceso']=proceso
tabla_logs['dat']=dat
tabla_logs['fecha_ejecucion']=fecha_log
tabla_logs['hora_inicio']=hora_log_inicio
tabla_logs['hora_fin']=hora_log_fin
tabla_logs['faltante']=tabla_logs['esperado']-tabla_logs['obtenido']
tabla_logs['codigo']=cod_proceso
tabla_logs['fecha_ini']=fecha
tabla_logs['fecha_ter']=fecha_fin
tabla_logs['id_afectado']=id_afectado
nuevas_columnas = ['codigo', 'proceso', 'dat', 'fecha_ejecucion', 'hora_inicio','hora_fin', 'dim', 'fecha_ini','fecha_ter','esperado', 'obtenido', 'faltante','falla', 'id_afectado']
tabla_logs = tabla_logs.reindex(columns=nuevas_columnas)

In [92]:
tabla_logs

,codigo,proceso,dat,fecha_ejecucion,hora_inicio,hora_fin,dim,fecha_ini,fecha_ter,esperado,obtenido,faltante,falla,id_afectado
0,13,DAT,dat_ceqx007_essi,2023-07-04,09:55,09:57,dim_oricas,2023-05-01,2024-12-31,100216,100216,0,0,id_oricas
1,13,DAT,dat_ceqx007_essi,2023-07-04,09:55,09:57,dim_cas,2023-05-01,2024-12-31,100216,100216,0,0,id_cas
2,13,DAT,dat_ceqx007_essi,2023-07-04,09:55,09:57,dim_red,2023-05-01,2024-12-31,100216,100216,0,0,id_red
3,13,DAT,dat_ceqx007_essi,2023-07-04,09:55,09:57,dim_cps,2023-05-01,2024-12-31,100216,100216,0,0,id_cps
4,13,DAT,dat_ceqx007_essi,2023-07-04,09:55,09:57,dim_cps,2023-05-01,2024-12-31,100216,100216,0,0,id_paqqui
5,13,DAT,dat_ceqx007_essi,2023-07-04,09:55,09:57,dim_pacsec,2023-05-01,2024-12-31,100216,100216,0,0,id_pacsec


In [93]:
tabla_logs.to_sql(name=f'logs_borrar1', con=connection4, if_exists='append', index=False,chunksize=10000)

6

In [94]:
finproceso=time.time()
tiempoproceso=finproceso - inicioProceso
tiempoproceso=round(tiempoproceso,3) #Redondea el tiempo de proceso a 3 decimales.
print("Proceso 1.3 completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Proceso 1.3 completado satisfactoriamente en 53.131 segundos


In [95]:
connection1.close()
connection2.close()
connection3.close()
connection4.close()

In [96]:
engine1.dispose()
engine2.dispose()
engine3.dispose()
engine4.dispose()